# Mini Project 4
Nama: Faraday Barr Fatahillah

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

AZURE_CHAT_DEPLOYMENT = os.getenv("AZURE_CHAT_DEPLOYMENT")
AZURE_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX = os.getenv("PINECONE_INDEX")

for name, value in {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
    "AZURE_CHAT_DEPLOYMENT": AZURE_CHAT_DEPLOYMENT,
    "AZURE_EMBEDDING_DEPLOYMENT": AZURE_EMBEDDING_DEPLOYMENT,
    "PINECONE_API_KEY": PINECONE_API_KEY
}.items():
    print(f"{name:30s} -> {'OK' if value else 'MISSING'}")

AZURE_OPENAI_API_KEY           -> OK
AZURE_OPENAI_ENDPOINT          -> OK
AZURE_OPENAI_API_VERSION       -> OK
AZURE_CHAT_DEPLOYMENT          -> OK
AZURE_EMBEDDING_DEPLOYMENT     -> OK
PINECONE_API_KEY               -> OK


In [3]:
print(AZURE_OPENAI_API_KEY)
print(AZURE_OPENAI_ENDPOINT)
print(AZURE_OPENAI_API_VERSION)

print(AZURE_CHAT_DEPLOYMENT)
print(AZURE_EMBEDDING_DEPLOYMENT)

print(PINECONE_API_KEY)
print(PINECONE_INDEX)

DQNPlPmePa5PHfgf0eddkfK1Rlf6nUKlOBR5bHGFvDBzShKwP5LqJQQJ99CEACHYHv6XJ3w3AAAAACOGOGF0
https://group2-bobeheus2.openai.azure.com/openai/v1
2024-02-01
o4-mini
text-embedding-3-small
pcsk_5iLHcg_KXNuiU1tLHLNpbPiLFwsn2nVsU7a1AKyfPP4vzW4X9vU4GfrZEBm4aHSJZFVA8J
mitsubishi-ai-chat


In [2]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

chat_llm = AzureChatOpenAI(
    azure_deployment=AZURE_CHAT_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
    max_completion_tokens=1024 
)

embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_EMBEDDING_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

test_llm = chat_llm.invoke(input="Who is Michael Faraday?")
print(f"Answer: {test_llm.content}")

test_embed = embeddings.embed_query("Mitsubishi")
print(f"Embedding dimension: {len(test_embed)}")

c:\Users\bobe\anaconda3\envs\bootcamp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    path="./data_mobil",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    use_multithreading=True
)

raw_docs = loader.load()
print(f"Total dokumen: {len(raw_docs)}")
print([f"  - {doc.metadata["source"]} ({len(doc.page_content)} chars)" for doc in raw_docs])

In [ ]:
for doc in raw_docs:
    filename = os.path.basename(doc.metadata["source"]).lower()
    name_clean = filename.replace(".txt", "")

    if "spec" in name_clean:
        doc.metadata["doc_type"] = "spec"
        model_name = name_clean.replace("_spec", "").replace("spec_", "")
    elif "review" in name_clean:
        doc.metadata["doc_type"] = "review"
        model_name = name_clean.replace("_review", "").replace("review_", "")
   
    doc.metadata["model_name"] = model_name.strip("_")
    doc.metadata["brand"] = "mitsubishi"

print("Contoh metadata setelah enrichment:")
for doc in raw_docs[:3]:
    print(f"   {doc.metadata}")
    

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter (
    chunk_size=500,
    chunk_overlap=75,
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len
)

chunks = splitter.split_documents(raw_docs)
print(f"Total chunks: {len(chunks)}")
print(f"Dari {len(raw_docs)} dokumen rata-rata {len(chunks)//len(raw_docs)} chunk/dokumen")
print(f"\nSample chunk ke-1:")
print(f"Metadata : {chunks[0].metadata}")
print(f"Content  : {chunks[0].page_content[:200]}...")

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

if PINECONE_INDEX not in pc.list_indexes().names():
    pc.create_index(
    name=PINECONE_INDEX, 
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

pine_index = pc.Index(PINECONE_INDEX)

In [ ]:
from langchain_pinecone import PineconeVectorStore
import time

BATCH_SIZE = 50

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks[:BATCH_SIZE],
    embedding=embeddings,
    index_name=PINECONE_INDEX
)

for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    vectorstore.add_documents(batch)
    print(f"Batch {i//BATCH_SIZE + 1}: {len(batch)} chunks uploaded")
    time.sleep(5)

print(f"Total {len(chunks)} saved di Pinecone")

In [ ]:
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX,
    embedding=embeddings
)

test_query = "Spesifikasi Mitsubishi X-Force"
results = vectorstore.similarity_search_with_score(test_query, k=3)

for i, (doc, score) in enumerate(results):
    print(f"[{i+1}] Test Query: {test_query}")
    print(f"Model  : {doc.metadata.get('model_name', 'unknown')}")
    print(f"Type   : {doc.metadata.get('doc_type', 'unknown')}")
    print(f"Content: {doc.page_content[:120]}...")

In [ ]:
from typing import List, Tuple, Dict
from rank_bm25 import BM25Okapi

def dense_search(query: str, k: int = 10) -> List[Tuple[any, float]]:
    """Semantic search menggunakan vector embedding."""
    return vectorstore.similarity_search_with_score(query, k=k)

def bm25_search(query: str, all_chunks: List, k: int = 10) -> List[Tuple[int, float]]:
    """Keyword search menggunakan BM25."""
    tokenized_corpus = [doc.page_content.lower().split() for doc in all_chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(idx, scores[idx]) for idx in top_indices]

def reciprocal_rank_fusion(
    dense_results: List[Tuple],
    bm25_results: List[Tuple],
    all_chunks: List,
    k: int = 60,
    top_n: int = 5
) -> List[any]:
    """Gabungkan hasil dense + BM25 dengan formula RRF."""
    rrf_scores: Dict[str, float] = {}
    doc_map: Dict[str, any] = {}

    for rank, (doc, _score) in enumerate(dense_results, start=1):
        doc_id = doc.page_content[:50]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)
        doc_map[doc_id] = doc

    for rank, (idx, _score) in enumerate(bm25_results, start=1):
        doc = all_chunks[idx]
        doc_id = doc.page_content[:50]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)
        doc_map[doc_id] = doc

    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [doc_map[doc_id] for doc_id in sorted_ids[:top_n]]


def hybrid_search(query: str, all_chunks: List, top_n: int = 5) -> List[any]:
    """Main hybrid search: dense + BM25 → RRF re-ranking."""
    dense_results = dense_search(query, k=10)
    bm25_results  = bm25_search(query, all_chunks, k=10)
    reranked_docs = reciprocal_rank_fusion(dense_results, bm25_results, all_chunks, top_n=top_n)
    return reranked_docs

In [ ]:
test_query = "kapasitas mesin dan torsi Xpander"
hybrid_results = hybrid_search(test_query, chunks, top_n=5)

print(f"Hybrid Search — Query: '{test_query}'")
print(f"{len(hybrid_results)} dokumen terpilih setelah RRF re-ranking\n")

for i, doc in enumerate(hybrid_results):
    print(f"[{i+1}]\nModel  : {doc.metadata.get('model_name', '?')} ")
    print(f"Type   : {doc.metadata.get('doc_type', '?')}")
    print(f"Content: {doc.page_content[:150]}...\n\n")

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are an AI assistant who is an expert on Mitsubishi cars.
     Answer users' questions ONLY based on the context of the provided document.
     If the information is not in the context, say, “That information is not available in my document.”
     Provide clear, accurate, and easy-to-understand answers in Indonesian.
     Include technical details if relevant."""),
    ("human",
     """Konteks dokumen:
        {context}

        Pertanyaan: {question}""")
])

def format_docs(docs: List) -> str:
    formatted = []
    for i, doc in enumerate(docs):
        model = doc.metadata.get('model_name', 'unknown').upper()
        dtype = doc.metadata.get('doc_type', 'general')
        formatted.append(f"[Sumber {i+1} — {model} {dtype}]\n{doc.page_content}")
    return "\n\n".join(formatted)

def hybrid_retriever(query: str) -> List:
    return hybrid_search(query, chunks, top_n=5)

rag_chain = (
    RunnableParallel(
        context  = RunnableLambda(hybrid_retriever) | RunnableLambda(format_docs),
        question = RunnablePassthrough()
    )
    | RAG_PROMPT
    | chat_llm
    | StrOutputParser()
)

def rag_with_sources(query: str) -> Dict:
    retrieved_docs = hybrid_retriever(query)
    context = format_docs(retrieved_docs)

    answer = (
        RAG_PROMPT
        | chat_llm
        | StrOutputParser()
    ).invoke({"context": context, "question": query})

    return {
        "query": query,
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "context": context,
    }

In [ ]:
test_q = "Apa saja fitur utama Mitsubishi Xpander?"
result = rag_with_sources(test_q)

print(f"Pertanyaan: {result['query']}")
print(f"\nJawaban:\n{result['answer']}")
print(f"\nDokumen sumber ({len(result['retrieved_docs'])} chunk):")
for doc in result['retrieved_docs']:
    print(f"   - {doc.metadata.get('model_name')} [{doc.metadata.get('doc_type')}]")

In [ ]:
FIVE_QUESTIONS = [
    "Berapa kapasitas mesin dan tenaga yang dihasilkan Mitsubishi Xpander?",
    "Apa perbedaan fitur keselamatan antara Xpander dan Pajero Sport?",
    "Bagaimana ulasan pengguna tentang konsumsi bahan bakar Mitsubishi Xpander?",
    "Apa saja varian yang tersedia untuk Mitsubishi Pajero Sport dan perbedaannya?",
    "Berdasarkan review, apa kelebihan dan kekurangan utama Mitsubishi Xpander?",
]

qa_results = []

print("=" * 70)
print("  🚗 MITSUBISHI KNOWLEDGE ASSISTANT — 5 PERTANYAAN")
print("=" * 70)

for i, question in enumerate(FIVE_QUESTIONS, 1):
    print(f"\n[{i}/5] {question}")
    print("-" * 60)

    result = rag_with_sources(question)
    qa_results.append(result)

    print(f"{result['answer']}")
    print(f"\nSumber: ", end="")
    sources = [f"{d.metadata.get('model_name')}[{d.metadata.get('doc_type')}]"
               for d in result['retrieved_docs'][:3]]
    print(", ".join(set(sources)))
    print("=" * 70)

In [ ]:
import json

def evaluate_faithfulness(context: str, answer: str) -> Dict:
    messages = [(
        "system", """Kamu adalah evaluator RAG system.
        Tugasmu: evaluasi apakah setiap klaim dalam JAWABAN didukung oleh KONTEKS.

        Return HANYA JSON valid (tanpa markdown):
        {"score": <0.0-1.0>, "supported_claims": <jumlah klaim didukung>,
        "total_claims": <total klaim>, "reasoning": "<penjelasan singkat>"}
         
         """),
        ("human", f"KONTEKS:\n{context}\n\nJAWABAN:\n{answer}")
    ]
    prompt = ChatPromptTemplate.from_messages(messages)
    response = (prompt | chat_llm | StrOutputParser()).invoke({})
    try:
        clean = response.strip().replace("```json", "").replace("```", "")
        return json.loads(clean)
    except Exception:
        return {"score": 0.0, "error": "parse failed", "raw": response[:200]}


def evaluate_answer_relevance(query: str, answer: str) -> Dict:
    """RAGAS Answer Relevance: seberapa relevan jawaban dengan pertanyaan?"""
    messages = [(
            "system", """Kamu adalah evaluator RAG system.
            Tugasmu: evaluasi seberapa relevan JAWABAN terhadap PERTANYAAN.

            Return HANYA JSON valid (tanpa markdown):
            {"score": <0.0-1.0>, "is_complete": <true/false>,
            "is_on_topic": <true/false>, "reasoning": "<penjelasan singkat>"}
        """),
        ("human", f"PERTANYAAN:\n{query}\n\nJAWABAN:\n{answer}")
    ]
    prompt = ChatPromptTemplate.from_messages(messages)
    response = (prompt | chat_llm | StrOutputParser()).invoke({})
    try:
        clean = response.strip().replace("```json", "").replace("```", "")
        return json.loads(clean)
    except Exception:
        return {"score": 0.0, "error": "parse failed", "raw": response[:200]}


def evaluate_context_precision(query: str, retrieved_docs: List) -> Dict:
    docs_text = "\n".join(
        [f"Chunk {i+1}: {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
    )
    messages = [(
            "system", """Kamu adalah evaluator RAG system.
            Tugasmu: untuk setiap chunk yang di-retrieve, nilai apakah chunk tersebut
            BENAR-BENAR DIPERLUKAN untuk menjawab pertanyaan.

            Return HANYA JSON valid (tanpa markdown):
            {"precision": <0.0-1.0>, "assessments": [{"chunk": 1, "relevant": true, "reason": "..."}]}
        """),
        ("human", f"PERTANYAAN:\n{query}\n\nCHUNKS:\n{docs_text}")
    ]
    prompt = ChatPromptTemplate.from_messages(messages)
    response = (prompt | chat_llm | StrOutputParser()).invoke({})
    try:
        clean = response.strip().replace("```json", "").replace("```", "")
        return json.loads(clean)
    except Exception:
        return {"precision": 0.0, "error": "parse failed", "raw": response[:200]}


def full_ragas_evaluation(query: str, top_k: int = 5) -> Dict:
    result = rag_with_sources(query)

    print(f"\n📝 Query   : {query}")
    print(f"💬 Answer  : {result['answer'][:200]}...")
    print(f"\n{'='*60}")
    print("📊 RAGAS EVALUATION")
    print(f"{'='*60}")

    faith = evaluate_faithfulness(result['context'], result['answer'])
    rel   = evaluate_answer_relevance(query, result['answer'])
    prec  = evaluate_context_precision(query, result['retrieved_docs'])

    f_score = faith.get('score', 0)
    r_score = rel.get('score', 0)
    p_score = prec.get('precision', 0)

    print(f"Faithfulness      : {f_score:.2f}  — {faith.get('reasoning', '')[:80]}")
    print(f"Answer Relevance  : {r_score:.2f}  — {rel.get('reasoning', '')[:80]}")
    print(f"Context Precision : {p_score:.2f}")
    print(f"{'='*60}")

    return {
        "query": query,
        "answer": result['answer'],
        "retrieved_docs": result['retrieved_docs'],
        "faithfulness": f_score,
        "answer_relevance": r_score,
        "context_precision": p_score,
    }

In [ ]:
eval_results = []

print("Memulai evaluasi RAGAS untuk 5 pertanyaan\n")

for i, q in enumerate(FIVE_QUESTIONS, 1):
    print(f"\n[{i}/5] Evaluasi: {q[:60]}...")
    eval_result = full_ragas_evaluation(q)
    eval_results.append(eval_result)
    time.sleep(2)

In [ ]:
print("\n" + "=" * 90)
print(f"{'RAGAS EVALUATION SUMMARY — MITSUBISHI RAG SYSTEM':^90}")
print("=" * 90)
print(f"  {'No':<4} {'Pertanyaan':<45} {'Faith':>7} {'Relev':>7} {'Prec':>7}")
print("-" * 90)

avg_faith = avg_rel = avg_prec = 0

for i, r in enumerate(eval_results, 1):
    q_short = r['query'][:43] + ".." if len(r['query']) > 43 else r['query']
    f = r['faithfulness']
    rv = r['answer_relevance']
    p = r['context_precision']
    avg_faith += f
    avg_rel   += rv
    avg_prec  += p
    print(f"  {i:<4} {q_short:<45} {f:>7.2f} {rv:>7.2f} {p:>7.2f}")

n = len(eval_results)
print("-" * 90)
print(f"  {'RATA-RATA':<49} {avg_faith/n:>7.2f} {avg_rel/n:>7.2f} {avg_prec/n:>7.2f}")
print("=" * 90)
print()
print("📌 Interpretasi skor (0.0 = buruk, 1.0 = sempurna):")
print("   Faithfulness      : Apakah jawaban didukung oleh konteks (anti-halusinasi)")
print("   Answer Relevance  : Apakah jawaban menjawab pertanyaan dengan tepat")
print("   Context Precision : Apakah dokumen yang di-retrieve memang relevan")

In [ ]:
YOUR_QUESTION = "Bagaimana perbandingan harga dan fitur antara Xpander dan Xpander Cross?"

result = rag_with_sources(YOUR_QUESTION)

print(f"❓ Pertanyaan  : {result['query']}")
print(f"\n💡 Jawaban:\n{result['answer']}")
print(f"\n📚 Sumber dokumen yang digunakan:")
for i, doc in enumerate(result['retrieved_docs'], 1):
    m = doc.metadata
    print(f"   [{i}] {m.get('model_name','?').upper()} — {m.get('doc_type','?')} "
          f"| '{doc.page_content[:80]}...'")